# Train `lstm_modelr` on Kaggle (30 GB RAM — full 280-day window)

## Setup (one time)
1. Kaggle → *Datasets* → **New Dataset** → upload BOTH zips
   (`js_colab_bundle.zip`, `js_pool_1318_1698.zip`) → name it e.g.
   `js-regen-pool` → keep it **Private**. Kaggle auto-extracts zips.
2. New Notebook → *Settings*: Accelerator **GPU** (P100 or T4 x2),
   Internet **on** (for pip), attach the dataset (Add Input).
3. Set `DATASET` below to your dataset path, Run All.

Outputs land in `/kaggle/working/out` — download
`preds/lstm_modelr_lstm_modelr_online.npz` + the `.json` when done.

In [ ]:
DATASET = '/kaggle/input/js-regen-pool'   # <- your dataset mount

import shutil, os, glob
print(os.listdir(DATASET))
# copy to local disk (input mounts are network storage)
shutil.copytree(f'{DATASET}/js_pool_1318_1698', '/kaggle/tmp/js_pool',
                dirs_exist_ok=True)
for d in ('src', 'scripts'):
    shutil.copytree(f'{DATASET}/{d}', f'/kaggle/working/project/{d}',
                    dirs_exist_ok=True)
print('data + code staged')

In [ ]:
%pip install --quiet 'polars>=1.20' 'pyarrow>=15' 'tqdm>=4.66'
import torch
assert torch.cuda.is_available(), 'enable GPU in notebook settings'
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
import os
os.environ['PYTHONPATH'] = '/kaggle/working/project/src'
!cd /kaggle/working/project && python -u scripts/train_from_memmap.py \
    --data /kaggle/tmp/js_pool \
    --model lstm_modelr --resample-mode window \
    --train-lo 1318 --train-hi 1597 --valid-lo 1599 --valid-hi 1698 \
    --device cuda --seed 42 --tag lstm_modelr \
    --batch-size 8 --epochs 30 --patience 5 \
    --out /kaggle/working/out

In [ ]:
import json
print(json.dumps(json.load(open('/kaggle/working/out/lstm_modelr_lstm_modelr.json')), indent=2))
# files persist under /kaggle/working -> Output tab -> download